# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset: this will download and parse the schema metadata
dataset = mlc.Dataset(url)

# Print high-level metadata
print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\nPublished: {dataset.metadata.datePublished}")
print(f"Available fields in metadata: {[attr for attr in dir(dataset.metadata) if not attr.startswith('_')]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

- Each *record set* in the Croissant schema describes a logical table or resource with its own fields and columns. 
- Fields are uniquely referenced by their `@id` attributes (e.g., `cr:Field/pI`).

**List available record sets and their fields:**

In [ ]:
# List all available record sets and their fields (by @id)
record_set_ids = [rs['@id'] for rs in getattr(dataset.metadata, 'recordSet', [])]
print(f"Found {len(record_set_ids)} record sets.")
for rs in getattr(dataset.metadata, 'recordSet', []):
    print(f"- Record Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    # field may be dict or list
    if isinstance(fields, dict):
        field_ids = [fields['@id']]
    else:
        field_ids = [f['@id'] for f in fields]
    print(f"    Field @ids: {field_ids}")
    # Also, columns (these may map to data file column names)
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        column_ids = [columns['@id']]
    else:
        column_ids = [c['@id'] for c in columns] if columns else []
    print(f"    Column @ids: {column_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below we will load all records from the first available record set.

In [ ]:
# Extract data from each record set
if not record_set_ids:
    raise RuntimeError("No record sets found in the dataset metadata.")

dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set {rs_id} (columns: {dataframes[rs_id].columns.tolist()})")
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

### Preview the Data
Preview the loaded DataFrame from the first available record set.

In [ ]:
# Choose the first record set for demonstration
main_record_set = record_set_ids[0]
df = dataframes[main_record_set]
print(f"First 5 rows from record set {main_record_set}:")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

> **Tip:** Use field `@id`s and/or column names listed above.

In [ ]:
# For illustration, select a numeric field (e.g., field with peptide count or MW)
# We'll guess the field/column name if column names are ambiguous.
import numpy as np

print(f"DataFrame columns: {df.columns.tolist()}")

# Try to auto-select a numeric column (fallback to user input if step-by-step)
numeric_candidates = [col for col in df.columns if df[col].dtype in [np.int64, np.float64] or pd.api.types.is_numeric_dtype(df[col])]

if not numeric_candidates:
    # Fallback: try to convert fields by name
    probable_fields = [c for c in df.columns if ("peptide" in c.lower()) or ("count" in c.lower()) or ("abund" in c.lower()) or ("mw" in c.lower()) or ("modification" in c.lower())]
    for col in probable_fields:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric columns (coerced): {numeric_candidates}")
else:
    print(f"Numeric columns: {numeric_candidates}")

if not numeric_candidates:
    raise RuntimeError("No numeric field found in this record set.")

# Pick the first candidate
numeric_field = numeric_candidates[0]
print(f"Using numeric field for filtering/EDA: '{numeric_field}'")

threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f} (N={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to select a group/categorical field for groupby
group_field = None
category_candidates = [col for col in df.columns if df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col])]
for cand in category_candidates:
    nunique = df[cand].nunique()
    if 2 < nunique < len(df) // 2:
        group_field = cand
        break
if group_field:
    print(f"Grouping by categorical field: '{group_field}'")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Mean {numeric_field} by {group_field}:")
    display(grouped_df.head())
else:
    print('No suitable categorical/group field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
- Distribution of the numeric field
- Boxplot/grouped means if there is a categorical field

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if group_field:
    plt.figure(figsize=(8, 4))
    order = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).index
    sns.boxplot(x=group_field, y=numeric_field, data=df, order=order)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library.

**Key findings:**
- Identified available record sets, fields, and columns using Croissant metadata.
- Extracted and previewed the main data table.
- Applied basic filtering, normalization, and aggregation on selected numeric fields.
- Visualized the distribution and grouped statistics.

Further analysis can leverage the detailed schema to link additional record sets or perform domain-specific statistics.

For more on Croissant datasets, see the [mlcroissant documentation](https://github.com/mlcommons/croissant).